# 📝 Text Summarisation using NLP – Google Colab Fine-Tuning

**B.Tech (AI & ML) Minor Project**

This notebook fine-tunes BART and T5 on the CNN/DailyMail dataset using a **free Colab GPU**.

### Steps
1. Enable GPU: Runtime → Change runtime type → GPU (T4)
2. Clone the repository and install dependencies
3. Fine-tune BART and T5
4. Download the fine-tuned models
5. Run evaluation

In [ ]:
# Step 1: Check GPU availability
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected. Go to Runtime → Change runtime type → GPU')

In [ ]:
# Step 2a: Clone the project repository
# Replace with your actual GitHub URL
!git clone https://github.com/YOUR_USERNAME/text-summarisation-nlp.git
%cd text-summarisation-nlp

In [ ]:
# Step 2b: Install all dependencies
!pip install -r requirements.txt -q
print('✓ Dependencies installed')

In [ ]:
# Step 3: Update config.yaml for GPU training (larger subsets + fp16)
import yaml

with open('config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

# Use larger subsets for GPU training
cfg['dataset']['subset'] = {'train': 10000, 'validation': 500, 'test': 500}

# Enable fp16 and larger batch sizes
cfg['training']['fp16'] = True
cfg['training']['per_device_train_batch_size'] = 8
cfg['training']['per_device_eval_batch_size'] = 8
cfg['training']['num_train_epochs'] = 3

with open('config.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('✓ config.yaml updated for GPU training')
print(f'  Train samples : {cfg["dataset"]["subset"]["train"]}')
print(f'  Batch size    : {cfg["training"]["per_device_train_batch_size"]}')
print(f'  fp16          : {cfg["training"]["fp16"]}')
print(f'  Epochs        : {cfg["training"]["num_train_epochs"]}')

In [ ]:
# Step 4a: Fine-tune BART
# Expected time on T4 GPU: ~30–45 min for 10k samples, 3 epochs
!python -m src.train --model bart

In [ ]:
# Step 4b: Fine-tune T5
# Expected time on T4 GPU: ~20–30 min for 10k samples, 3 epochs
!python -m src.train --model t5

In [ ]:
# Step 5a: Run extractive models on test set
!python scripts/run_extractive.py

In [ ]:
# Step 5b: Run fine-tuned abstractive models on test set
!python scripts/run_abstractive.py

In [ ]:
# Step 5c: Evaluate all models
!python scripts/run_evaluation.py

In [ ]:
# Step 6: View results
import pandas as pd
from IPython.display import Image, display

# Metrics table
df = pd.read_csv('results/metrics.csv')
print('=== Model Comparison Metrics ===')
display(df.style.highlight_max(subset=df.columns[1:], color='lightgreen'))

# Chart
display(Image('results/comparison.png'))

In [ ]:
# Step 7: Download fine-tuned models to your local machine
# This creates zip archives of the model checkpoints
import zipfile, os

for model_dir in ['models/bart_finetuned', 'models/t5_finetuned']:
    if os.path.exists(model_dir):
        zip_name = model_dir.replace('/', '_') + '.zip'
        with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
            for root, dirs, files in os.walk(model_dir):
                for file in files:
                    filepath = os.path.join(root, file)
                    zf.write(filepath)
        print(f'Created {zip_name}')

# Download from Colab
from google.colab import files
for f_name in ['models_bart_finetuned.zip', 'models_t5_finetuned.zip']:
    if os.path.exists(f_name):
        files.download(f_name)

# Also download results
!zip -r results.zip results/
files.download('results.zip')